<a href="https://colab.research.google.com/github/AsemaniJoshua/Animal-Disease-Models/blob/main/notebooks/train_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐐 Animal Disease Models: Mobile Anemia Detection Pipeline
### Fast 1-Click GPU Training & TFLite Mobile Export for Sheep & Goat Anemia Diagnosis

This Google Colab notebook trains the two mobile AI models:
1. **Model 1: The 'Finder' (YOLOv8 Nano)** — Detects and isolates inner eyelids.
2. **Model 2: The 'Judge' (MobileNetV3 Small)** — Classifies anemia severity (Green / Yellow / Red).
3. **Exports directly to `.tflite`** for offline deployment in your mobile app.

---
### Step 0: Ensure GPU is Enabled
Go to **Runtime > Change runtime type** and select **T4 GPU**.

In [1]:
# Check GPU availability
!nvidia-smi

Tue Sep  8 18:20:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 1: Install Dependencies & Clone/Upload Datasets

In [2]:
!pip install -q ultralytics torchvision torch pandas openpyxl onnx onnxruntime
import os, sys
print("Dependencies installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 4.5 MB/s eta 0:00:00
Dependencies installed successfully!


### Step 2: Prepare Datasets
Run data preparation to organize YOLO eye detection and CP-AnemiC conjunctiva images.

In [3]:
# Run data preparation scripts
!python data/prepare_yolo_data.py
!python data/prepare_judge_data.py

python3: can't open file '/content/data/prepare_yolo_data.py': [Errno 2] No such file or directory
python3: can't open file '/content/data/prepare_judge_data.py': [Errno 2] No such file or directory


### Step 3: Train Model 1 (The Finder / YOLOv8 Nano Eye Detector)

In [4]:
from ultralytics import YOLO

model1 = YOLO("yolov8n.pt")
results1 = model1.train(
    data="configs/dataset_yolo.yaml",
    epochs=30,
    imgsz=640,
    batch=32,
    device=0,
    project="runs/finder",
    name="train",
    exist_ok=True
)
print("Model 1 (Finder) training complete!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)


KeyboardInterrupt: 

### Step 4: Train Model 2 (The Judge / MobileNetV3 Anemia Classifier)

In [ ]:
!python training/train_judge.py --epochs 30 --batch 32 --device cuda

### Step 5: Export Both Models to `.tflite` for Mobile

In [ ]:
import shutil
from pathlib import Path
from ultralytics import YOLO

# Export Model 1 to TFLite
print("Exporting Model 1 (Finder) to TFLite...")
finder_model = YOLO("runs/finder/train/weights/best.pt")
finder_tflite = finder_model.export(format="tflite", imgsz=640, int8=True)
shutil.copy2(finder_tflite, "exported_models/finder_yolo.tflite")

# Export Model 2 to ONNX and TFLite
!python export/export_judge.py

print("Exported files in exported_models/:")
!ls -lh exported_models

### Step 6: Test End-to-End Simulation

In [ ]:
!python pipeline/end_to_end_demo.py --lang hausa

### Step 7: Download Models for Mobile App
Zip and download your `.tflite` models directly to copy into your mobile repo's `assets/models/` folder.

In [ ]:
from google.colab import files
!zip -r mobile_models.zip exported_models/
files.download("mobile_models.zip")
print("Download started! Extract mobile_models.zip and place into your mobile app repo assets.")